# Kaggle Autonomous Multi-Agent System (KAMAS)
## Competition: Playground Series Season 6 Episode 9 (`playground-series-s6e9`)
### Strategic Breakthrough to 0.946+ (Original Dataset + Neural Net + Ensembling)

This pipeline executes the 3 Grandmaster techniques to break through the 0.942 plateau:
1. **Ground-Truth Data Augmentation:** 10,000 original physical samples (`itzzomkar/ev-adoption-behavior-and-range-anxiety`) injected strictly into training folds.
2. **Bayesian Target Encoding & Ratios:** Infrastructure density, commute ratios, economic cushioning, and subsidy gating.
3. **Orthogonal Model Diversity:** LightGBM, CatBoost (GPU), XGBoost (CUDA), and PyTorch Tabular Neural Network.
4. **Phase 10 Blender:** Caruana Hill Climbing and Nelder-Mead metric optimization on ROC-AUC.

> **Recommended Kaggle Settings:** Accelerator = **GPU T4 x2**, Internet = **ON**.

### 1. Synchronize Repository & Environment

In [ ]:
import os
import sys
from pathlib import Path

REPO_NAME = "electric-vehicle"
repo_path = Path(f"/kaggle/working/{REPO_NAME}")

if repo_path.exists():
    %cd /kaggle/working/electric-vehicle
    !git pull origin main
else:
    %cd /kaggle/working
    !git clone https://github.com/tuboa2/electric-vehicle-purchase-predictor.git electric-vehicle
    %cd /kaggle/working/electric-vehicle

sys.path.insert(0, str(Path.cwd()))
print(f"[*] Ready in: {Path.cwd()}")

### 2. Verify Hardware Acceleration & Dependencies

In [ ]:
!nvidia-smi
import torch
import lightgbm as lgb
import catboost as cb
import xgboost as xgb

print(f"[+] CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[+] GPU: {torch.cuda.get_device_name(0)}")

### 3. Model 1: LightGBM with Original Dataset & Target Encoding
Trains 5-fold LightGBM with domain features, target encoding, and the 10,000 ground-truth samples.

In [ ]:
!python scripts/kaggle_train.py --model lgbm --features domain --use-original

### 4. Model 2: CatBoost with GPU Acceleration & Original Dataset
Trains 5-fold CatBoost with symmetric oblivious trees on GPU.

In [ ]:
!python scripts/kaggle_train.py --model catboost --features domain --use-original

### 5. Model 3: XGBoost with CUDA Acceleration & Original Dataset
Trains 5-fold XGBoost utilizing histogram tree method on GPU.

In [ ]:
!python scripts/kaggle_train.py --model xgboost --features domain --use-original

### 6. Model 4: PyTorch Tabular Neural Network (MLP with Entity Embeddings)
Trains a deep tabular network on GPU to generate non-axis-aligned, orthogonal predictions.

In [ ]:
!python scripts/kaggle_train_nn.py --epochs 12 --batch-size 2048

### 7. Phase 10: Run the Multi-Model Ensemble Blender
Combines predictions across LightGBM, CatBoost, XGBoost, and PyTorch Tabular MLP via Nelder-Mead and Rank Averaging.

In [ ]:
!python scripts/kaggle_blend.py

### 8. Inspect Final Submission File

In [ ]:
import pandas as pd
sub_file = Path("/kaggle/working/submission.csv")
if sub_file.exists():
    df = pd.read_csv(sub_file)
    print(f"[+] File size: {sub_file.stat().st_size / 1024 / 1024:.2f} MB")
    print(f"[+] Row count: {len(df):,}")
    print(f"[+] Nulls:     {df.isnull().sum().to_dict()}")
    print("\nFirst 10 rows:")
    print(df.head(10))
else:
    print("[!] submission.csv not found!")